# Camada Silver - DataCo Smart Supply Chain (Diogo)

Limpa a tabela Bronze e grava a tabela Silver em Delta, na base `silver` do grupo.

| Tabela Silver | 1 linha = | Origem (Bronze) | Usada no modelo em |
|---|---|---|---|
| `silver.dataco_itens` | 1 item de encomenda (`order_item_id`) com encomenda entre junho e dezembro de 2017 | `dataco_supply_chain` | `fact_encomendas_dataco`, `dim_perfil_encomenda`, `dim_localizacao`, `dim_categoria_produto` |

Regras do grupo aplicadas:
- **Intervalo comum:** só entram encomendas de 2017-06-01 a 2017-12-31 (antes estava janeiro a setembro).
- A partir de outubro de 2017 as encomendas passam a ter um só item. A secção "Período" mostra os números por mês;
  as análises usam métricas **por item** (vendas, margem e atraso por item), que não dependem do número de itens por encomenda.
- `order_item_discount` é um valor em dinheiro: no modelo chama-se `desconto_valor` (diferente do `desconto_pct` do Superstore).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Silver - DataCo (Diogo)") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
# Intervalo de tempo comum do grupo (README): só entram registos de 1 de junho a 31 de dezembro de 2017
DATA_INICIO = "2017-06-01"
DATA_FIM = "2017-12-31"

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver LOCATION 'hdfs://hdfs-nn:9000/warehouse/silver.db/'")

def gravar_silver(df, tabela):
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/silver.db/{tabela}/") \
        .saveAsTable(f"silver.{tabela}")
    print(f"silver.{tabela}: {spark.table(f'silver.{tabela}').count()} linhas")

def nulos(df):
    """Número de nulos por coluna."""
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

def texto(c):
    """Remove espaços nas pontas e passa texto vazio a NULL."""
    t = F.trim(F.col(c))
    return F.when(t == "", None).otherwise(t)

## Leitura da Bronze e tipos

A Bronze tem os nomes originais (com espaços e parênteses) e tudo como texto; aqui passam a snake_case e aos tipos certos.

In [4]:
import re
def to_snake(nome):
    nome = nome.replace("(DateOrders)", "").replace("(real)", "real").replace("(scheduled)", "scheduled")
    return re.sub(r"[^a-z0-9]+", "_", nome.strip().lower()).strip("_")

bronze = spark.table("bronze.dataco_supply_chain").drop("_data_ingestao", "_ficheiro_origem")
df = bronze.toDF(*[to_snake(c) for c in bronze.columns])
print(df.columns)

inteiros = ["days_for_shipping_real", "days_for_shipment_scheduled", "late_delivery_risk", "category_id", "customer_id",
            "department_id", "order_customer_id", "order_id", "order_item_cardprod_id", "order_item_id",
            "order_item_quantity", "product_card_id", "product_category_id", "product_status"]
decimais = ["benefit_per_order", "sales_per_customer", "order_item_discount", "order_item_discount_rate",
            "order_item_product_price", "order_item_profit_ratio", "sales", "order_item_total",
            "order_profit_per_order", "product_price", "latitude", "longitude"]
for c in inteiros:
    df = df.withColumn(c, F.col(c).cast("int"))
for c in decimais:
    df = df.withColumn(c, F.col(c).cast("double"))
print("Total de registos lidos:", df.count())      # 180519

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date', 'shipping_mode']
Total d

## Limpeza e transformação

(Mesma lógica da versão anterior do Diogo; mudou o período.)

In [5]:
# 1. Eliminar colunas (duplicadas, pessoais, vazias, constantes, deriváveis ou sem utilidade analítica)
#    Ficam order_country/order_state (local da encomenda), que é a localização usada no modelo.
colunas_eliminar = [
    "sales_per_customer", "benefit_per_order", "order_item_cardprod_id",
    "customer_fname", "customer_lname", "customer_street",
    "product_description", "product_status",
    "customer_email", "customer_password", "order_zipcode",
    "order_customer_id", "product_category_id", "product_image", "product_price",
    "latitude", "longitude", "order_item_discount_rate", "order_item_profit_ratio",
    "customer_city", "customer_state", "customer_zipcode", "customer_country",
    "delivery_status",
]
df_clean = df.drop(*colunas_eliminar)
print(f"Eliminadas {len(colunas_eliminar)} colunas. Restam {len(df_clean.columns)}.")

# 2. Converter datas (texto -> timestamp)
FMT = "M/d/yyyy H:mm"
df_clean = (df_clean
    .withColumn("order_date", F.to_timestamp("order_date", FMT))
    .withColumn("shipping_date", F.to_timestamp("shipping_date", FMT)))
antes = df_clean.count()
df_clean = df_clean.filter(F.col("order_date").isNotNull() & F.col("shipping_date").isNotNull())
print("Removidos por data inválida:", antes - df_clean.count())

# 3. Texto: espaços a mais e vazio -> NULL
for c, t in df_clean.dtypes:
    if t == "string":
        df_clean = df_clean.withColumn(c, F.regexp_replace(F.trim(F.col(c)), r"\s+", " "))
        df_clean = df_clean.withColumn(c, F.when(F.col(c) == "", None).otherwise(F.col(c)))

# 4. Cada category_name fica com um único category_id (o mais frequente)
w = Window.partitionBy("category_name").orderBy(F.col("n").desc(), F.col("category_id"))
cat_ok = (df_clean.groupBy("category_name", "category_id").agg(F.count("*").alias("n"))
            .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
            .select("category_name", F.col("category_id").alias("category_id_ok")))
df_clean = df_clean.join(cat_ok, "category_name", "left").drop("category_id").withColumnRenamed("category_id_ok", "category_id")

# 5. Dias reais (dias de calendário) e indicador de atraso (dias reais > dias agendados)
dias_reais = F.datediff(F.to_date("shipping_date"), F.to_date("order_date"))
print("Diferenças face aos dias reais originais:", df_clean.filter(dias_reais != F.col("days_for_shipping_real")).count())
df_clean = (df_clean
    .withColumn("days_for_shipping_real", dias_reais.cast("int"))
    .withColumn("late_delivery_risk", F.when(F.col("days_for_shipping_real") > F.col("days_for_shipment_scheduled"), 1).otherwise(0)))

# 6. Excluir encomendas canceladas e suspeitas de fraude
excl = F.col("order_status").isin("CANCELED", "SUSPECTED_FRAUD")
print("Removidas (canceladas/fraude):", df_clean.filter(excl).count(), "linhas")
df_clean = df_clean.filter(~excl)

Eliminadas 24 colunas. Restam 29.
Removidos por data inválida: 0
Diferenças face aos dias reais originais: 0
Removidas (canceladas/fraude): 7754 linhas


## Período

Antes de filtrar, os números por mês mostram a mudança de outubro de 2017: o número de itens por encomenda cai para 1,
mas as métricas por item (vendas por item, margem, atraso, categorias e mercados) permitem ver se os dados continuam utilizáveis.

In [6]:
diag = (df_clean.filter(F.to_date("order_date").between(DATA_INICIO, "2018-01-31"))
    .groupBy(F.date_format("order_date", "yyyy-MM").alias("mes"))
    .agg(F.count("*").alias("itens"),
         F.countDistinct("order_id").alias("encomendas"),
         F.round(F.count("*") / F.countDistinct("order_id"), 2).alias("itens_por_encomenda"),
         F.round(F.avg("sales"), 2).alias("vendas_por_item"),
         F.round(F.sum("order_profit_per_order") / F.sum("sales") * 100, 1).alias("margem_pct"),
         F.round(F.avg("late_delivery_risk") * 100, 1).alias("pct_atraso"),
         F.countDistinct("category_name").alias("n_categorias"),
         F.countDistinct("market").alias("n_mercados"))
    .orderBy("mes"))
diag.show(20)

+-------+-----+----------+-------------------+---------------+----------+----------+------------+----------+
|    mes|itens|encomendas|itens_por_encomenda|vendas_por_item|margem_pct|pct_atraso|n_categorias|n_mercados|
+-------+-----+----------+-------------------+---------------+----------+----------+------------+----------+
|2017-06| 4730|      1605|               2.95|         208.41|      10.7|      56.0|          24|         2|
|2017-07| 5052|      1693|               2.98|         207.32|      10.6|      57.7|          24|         1|
|2017-08| 5111|      1695|               3.02|          208.9|      11.9|      58.6|          24|         1|
|2017-09| 4931|      1640|               3.01|          220.7|      10.8|      57.1|          26|         1|
|2017-10| 2169|      2019|               1.07|          474.8|      10.9|      55.8|          20|         1|
|2017-11| 1956|      1956|                1.0|         306.13|      11.2|      57.9|           8|         2|
|2017-12| 2031|    

**Conclusão sobre o período.** De junho a setembro cada encomenda tem cerca de 3 itens; a partir de outubro passa a ter 1.
Mas a margem (11–13%) e a percentagem de itens atrasados (56–58%) mantêm-se iguais, por isso os dados de outubro a dezembro
são utilizáveis e entram na Silver (intervalo comum do grupo). Limitações a referir no relatório:
- as médias **por encomenda** (itens por encomenda, valor por encomenda) não são comparáveis antes e depois de outubro; usar métricas **por item**;
- de outubro a dezembro aparecem menos categorias por mês (8 a 20 em vez de 24), por isso as comparações por categoria nesses meses têm menos dados.

In [7]:
antes = df_clean.count()
df_clean = df_clean.filter(F.to_date("order_date").between(DATA_INICIO, DATA_FIM))
print(f"Removidos {antes - df_clean.count()} itens fora de {DATA_INICIO} a {DATA_FIM}. Ficam {df_clean.count()}.")

df_clean = df_clean.withColumn("data_encomenda", F.to_date("order_date"))

Removidos 146785 itens fora de 2017-06-01 a 2017-12-31. Ficam 25980.


## Validações

In [8]:
n_itens = df_clean.count()
assert n_itens == df_clean.select("order_item_id").distinct().count(), "order_item_id repetido"

colunas_kpi = ["order_id", "order_item_id", "sales", "order_profit_per_order", "market", "category_name",
               "shipping_mode", "order_status", "days_for_shipping_real", "days_for_shipment_scheduled", "late_delivery_risk"]
nul = df_clean.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas_kpi]).first().asDict()
assert sum(nul.values()) == 0, nul

# Atributos da encomenda iguais em todas as linhas da mesma encomenda
col_enc = ["order_status", "market", "order_region", "order_country", "shipping_mode", "late_delivery_risk",
           "days_for_shipping_real", "days_for_shipment_scheduled", "order_date", "shipping_date"]
variam = (df_clean.groupBy("order_id").agg(*[F.countDistinct(c).alias(c) for c in col_enc])
            .agg(*[F.max(c).alias(c) for c in col_enc]).first().asDict())
assert all(v == 1 for v in variam.values()), variam
assert df_clean.filter(~F.col("data_encomenda").between(DATA_INICIO, DATA_FIM)).count() == 0

print(f"Validações OK. Itens: {n_itens} | Encomendas: {df_clean.select('order_id').distinct().count()}")

Validações OK. Itens: 25980 | Encomendas: 12639


In [9]:
# Mercado: cada país pertence sempre ao mesmo mercado (Europe, LATAM, Pacific Asia), por isso a coluna está correta.
# Mas no DataCo cada mês só tem encomendas de 1 ou 2 mercados (jun: Europe+LATAM; jul-out: Europe; nov: Europe+Pacific Asia;
# dez: Pacific Asia). Comparar mercados é, na prática, comparar meses diferentes: é uma limitação a referir no relatório.
df_clean.groupBy(F.date_format("order_date", "yyyy-MM").alias("mes"), "market").count().orderBy("mes", "market").show()
df_clean.groupBy("market").agg(F.countDistinct("order_country").alias("paises"), F.collect_set("order_region").alias("regioes")).show(truncate=False)

# Países da encomenda: o DataCo usa nomes em espanhol (ex.: "Estados Unidos", "Francia").
# Ficam assim na Silver; a tradução para os nomes do World Data é feita na tabela de países do grupo (Gold).
df_clean.groupBy("order_country").count().orderBy(F.desc("count")).show(15, truncate=False)

+-------+------------+-----+
|    mes|      market|count|
+-------+------------+-----+
|2017-06|      Europe| 2036|
|2017-06|       LATAM| 2694|
|2017-07|      Europe| 5052|
|2017-08|      Europe| 5111|
|2017-09|      Europe| 4931|
|2017-10|      Europe| 2169|
|2017-11|      Europe|  644|
|2017-11|Pacific Asia| 1312|
|2017-12|Pacific Asia| 2031|
+-------+------------+-----+

+------------+------+---------------------------------------------------+
|market      |paises|regioes                                            |
+------------+------+---------------------------------------------------+
|Europe      |18    |[Northern Europe, Southern Europe, Western Europe] |
|Pacific Asia|23    |[South Asia, Southeast Asia, Eastern Asia, Oceania]|
|LATAM       |19    |[South America, Caribbean, Central America]        |
+------------+------+---------------------------------------------------+

+-------------+-----+
|order_country|count|
+-------------+-----+
|Francia      |5786 |
|Alemania     |

In [10]:
gravar_silver(df_clean, "dataco_itens")

silver.dataco_itens: 25980 linhas


## Resultado

In [11]:
spark.sql("""
    SELECT date_format(order_date, 'yyyy-MM') AS mes, COUNT(*) AS itens, COUNT(DISTINCT order_id) AS encomendas,
           ROUND(SUM(sales), 0) AS vendas, ROUND(AVG(late_delivery_risk) * 100, 1) AS pct_itens_atrasados
    FROM silver.dataco_itens GROUP BY 1 ORDER BY 1
""").show()
spark.sql("""
    SELECT shipping_mode, COUNT(*) AS itens, ROUND(AVG(late_delivery_risk) * 100, 1) AS pct_itens_atrasados
    FROM silver.dataco_itens GROUP BY shipping_mode ORDER BY shipping_mode
""").show()

+-------+-----+----------+---------+-------------------+
|    mes|itens|encomendas|   vendas|pct_itens_atrasados|
+-------+-----+----------+---------+-------------------+
|2017-06| 4730|      1605| 985768.0|               56.0|
|2017-07| 5052|      1693|1047375.0|               57.7|
|2017-08| 5111|      1695|1067685.0|               58.6|
|2017-09| 4931|      1640|1088272.0|               57.1|
|2017-10| 2169|      2019|1029832.0|               55.8|
|2017-11| 1956|      1956| 598789.0|               57.9|
|2017-12| 2031|      2031| 480846.0|               57.7|
+-------+-----+----------+---------+-------------------+

+--------------+-----+-------------------+
| shipping_mode|itens|pct_itens_atrasados|
+--------------+-----+-------------------+
|   First Class| 3895|              100.0|
|      Same Day| 1440|               56.0|
|  Second Class| 5123|               80.0|
|Standard Class|15522|               39.2|
+--------------+-----+-------------------+



In [12]:
spark.stop()